# Target Sanity Check

Step 0 of feature selection: do the target labels track the core laboratory
values? For DS1 and DS2, each core lab is summarised per target class (mean)
and split into quartile bands (share of each class inside each band). The
class-mean spread relative to the feature's standard deviation sizes the
signal: < 0.15 negligible, 0.15-0.5 weak, > 0.5 clear.

Read-only - writes nothing.

In [1]:
import json
from pathlib import Path

import pandas as pd

root = Path.cwd()
while not (root / "configs" / "config.json").is_file():
    if root == root.parent:
        raise FileNotFoundError("configs/config.json not found above the working directory")
    root = root.parent

CONFIG_PATH = root / "configs" / "config.json"
config = json.loads(CONFIG_PATH.read_text())


def dataset_file(ds_id, stage):
    """Path of one dataset file; config paths are relative to configs/."""
    ds = config["datasets"]["list"][ds_id]
    entry = ds["files"][stage]
    if isinstance(entry, dict):
        entry = entry.get("train", entry.get("data"))
    elif isinstance(entry, list):
        entry = entry[0]
    return root / "configs" / config["paths"]["datasets_dir"] / ds["folder"] / entry


def load_frame(ds_id, stage):
    ds = config["datasets"]["list"][ds_id]
    return pd.read_csv(dataset_file(ds_id, stage), na_values=ds["na_values"] or None)

In [2]:
CHECKS = {
    "DS1": ("cleaned", ["sc", "hemo", "rbcc"]),
    "DS2": ("original", [
        "Estimated Glomerular Filtration Rate (eGFR)",
        "Serum creatinine (mg/dl)",
        "Hemoglobin level (gms)",
    ]),
}


def sanity_check(ds_id):
    stage, labs = CHECKS[ds_id]
    ds = config["datasets"]["list"][ds_id]
    df = load_frame(ds_id, stage)
    y = df[ds["target"]["column"]]

    print(f"{ds_id} - {ds['name']}  ({len(df)} rows)")
    ratios = {}
    for lab in labs:
        s = df[lab]
        means = s.groupby(y).mean().round(2)
        spread = (means.max() - means.min()) / s.std()
        ratios[lab] = spread
        bands = pd.qcut(s, 4, duplicates="drop")
        share = (pd.crosstab(y, bands, normalize="index") * 100).round(1)
        table = pd.DataFrame({"mean": means})
        for b in share.columns:
            table[str(b)] = share[b]
        print(f"\n{lab}  (class-mean spread / std = {spread:.2f})")
        print(table.to_string())

    print("\nCONCLUSION")
    for lab, r in ratios.items():
        level = "clear" if r > 0.5 else "weak" if r > 0.15 else "negligible"
        print(f"- {lab}: separation {r:.2f} ({level})")
    best = max(ratios.values())
    if best > 0.5:
        verdict = "labels clearly track the core labs - feature selection is meaningful"
    elif best > 0.15:
        verdict = "labels only weakly track the core labs - expect a modest feature signal"
    else:
        verdict = "labels do NOT track the core labs - selected features risk being noise"
    print(f"- {ds_id} verdict: {verdict}")


for ds_id in ("DS1", "DS2"):
    print()
    sanity_check(ds_id)


DS1 - UCI Chronic Kidney Disease Dataset  (400 rows)

sc  (class-mean spread / std = 0.62)
        mean  (0.399, 0.9]  (0.9, 1.3]  (1.3, 2.8]  (2.8, 76.0]
class                                                          
ckd     4.41           8.4        13.9        38.7         39.1
notckd  0.87          57.2        42.8         0.0          0.0

hemo  (class-mean spread / std = 1.56)
         mean  (3.099, 10.3]  (10.3, 12.65]  (12.65, 15.0]  (15.0, 17.8]
class                                                                   
ckd     10.65           43.1           42.2           12.7           2.0
notckd  15.19            0.0            0.0           51.4          48.6

rbcc  (class-mean spread / std = 1.39)
        mean  (2.099, 3.9]  (3.9, 4.8]  (4.8, 5.4]  (5.4, 8.0]
class                                                         
ckd     3.95          54.0        33.3         9.5         3.2
notckd  5.38           0.0        22.4        34.3        43.4

CONCLUSION
- sc: separation